# Inf-SSM on Split-CIFAR100 (10 tasks x 10 classes): Vim-small backbone

Port of the official Inf-SSM code ([isaacNingLee/Inf-SSM](https://github.com/isaacNingLee/Inf-SSM), CVPR 2026) in
`baselines/inf-ssm`, with the paper's protocol and hyperparameters (Table 13 / `vim/scripts/cifar100_10task.sh`):
**Vim-small**, 10 tasks, **40 epochs per task**, batch 128, AdamW lr 1e-5 (x0.5 per task), cosine schedule,
mixup/cutmix, Inf-SSM lambda = 3e4, fp32. Task 0 is not trained: seed 0 starts from the authors' task-0
checkpoint ([ileening0022/Inf-SSM](https://huggingface.co/ileening0022/Inf-SSM)).

Paper result (Table 2, mean ± std over seeds 0/10/100): **AA 26.53 ± 3.10, AIA 54.24 ± 1.87, FM 24.00 ± 1.80**.
The protocol differs from Mamba-CL (shared 100-way head, all weights trained, 224 px, class-incremental eval),
so these numbers are not comparable with our Mamba-CL run.

Environment is the same as the Mamba-CL notebook: Python 3.11 venv, torch 2.3.1 + cu121, causal-conv1d 1.4.0,
mamba-ssm 1.2.2 (stock kernels; the Vim bidirectional mixer is rebuilt on top of them in `mamba_vim.py`).

**Settings:** GPU **T4 x2**, Internet on. One run uses both GPUs (`DATA_PARALLEL`): each batch of 128 is split 64 + 64.
`GRAD_CKPT` recomputes each Vim block in the backward pass so Vim-small fits a T4 (roughly 30% slower); try `False` if memory allows.
Attach these Kaggle Secrets (Add-ons > Secrets):
- `HF_TOKEN`: a Hugging Face token with **write** access. Checkpoints and `results.json` are mirrored to the private
  repo `HF_REPO`, so they survive the end of the session.
- `WANDB_API_KEY` (optional). Without it W&B logs offline; sync later with `wandb sync` on the saved `wandb/` folder.

**Resuming:** Kaggle stops sessions at 12 h. Start a new session and **Run All** with the same parameters. The run
pulls its checkpoints from `HF_REPO` and continues: from the last finished task (`task_end.pt`, `results.json`), or
inside a task from the last uploaded epoch (`epoch.pt` + `task_best.pt`, every `HF_SYNC_EVERY` epochs).

In [ ]:
# ---- run parameters ----
REPO     = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH   = "feat/inf-ssm-vim"
DATASET  = "cifar100"   # cifar100 | imagenet_r | caltech256
TASKS    = 10           # 5 or 10 (the paper hyperparameters for that split are picked automatically)
SEED     = 0            # paper: 0, 10, 100. Only seed 0 has a released task-0 checkpoint
EPOCHS   = 40           # paper
BATCH    = 128          # paper; with DATA_PARALLEL each T4 gets 64
CL_METHOD = "reg"       # reg = Inf-SSM, seq = plain sequential fine-tuning (lower bound)
FULL_RUN = SEED != 0    # also train task 0, from ImageNet Vim-small (needed for seeds without a released checkpoint)
FINETUNE = "vim_small_in1k" if FULL_RUN else "hf"
USE_AMP  = False        # paper runs fp32; True is faster on T4 but changes the numerics
DATA_PARALLEL = True    # split each batch across both T4s (one run); False = single GPU
GRAD_CKPT = True        # per-block activation recomputation, to fit Vim-small in 16 GB
WORKERS  = 4
WANDB_PROJECT = "inf-ssm-baseline"
PROGRESS_INTERVAL = 60  # seconds between progress-bar updates in the log

# ---- checkpoints ----
CKPT_DIR = "/kaggle/working/checkpoints"
HF_REPO  = "<hf-username>/hippocortex-checkpoints"   # private model repo, created on first run
HF_SYNC_EVERY = 5        # upload the mid-task checkpoint every N epochs (~0.4 GB each); 0 = only at task ends
HF_SQUASH = True         # squash HF_REPO's history after each task, so it keeps one copy per file (use a dedicated repo)
assert "<" not in HF_REPO, "set HF_REPO to your Hugging Face username/repo"

In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    # Fall back to main once the branch is merged (GitHub deletes merged branches here).
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO]).returncode != 0:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
VENV = "/kaggle/working/py311_env"
PY, PIP = f"{VENV}/bin/python", f"{VENV}/bin/pip"
!cd {ROOT} && git log --oneline -1

In [ ]:
# Python 3.11 venv (the Kaggle image's Python is too new for torch 2.3.1 / mamba-ssm 1.2.2)
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null
!python3.11 -m venv {VENV}
!{PY} -m ensurepip
!{PIP} install -q --upgrade pip
!{PIP} install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q ninja packaging wheel
!{PIP} install -q --no-build-isolation causal-conv1d==1.4.0
!{PIP} install -q --no-build-isolation mamba-ssm==1.2.2
!{PIP} install -q numpy==1.26.4 timm==1.0.9 einops==0.8.0 tqdm huggingface_hub && {PIP} install -q -U wandb
# The model must find the CUDA kernels; without them it falls back to the (very slow) PyTorch reference scan.
!cd {ROOT}/baselines/inf-ssm && {PY} -c "import torch, mamba_vim as m; assert m.selective_scan_fn is not None and m.causal_conv1d_fn is not None, 'mamba kernels missing'; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'GPU(s) | mamba + causal-conv1d kernels OK')"

In [ ]:
# Data. ImageNet-R / Caltech-256 are image folders; CIFAR-100 is fetched by torchvision.
DATA = {"cifar100": "/kaggle/working/data/cifar100",
        "imagenet_r": "/kaggle/working/data/imagenet-r",
        "caltech256": "/kaggle/working/data/256_ObjectCategories"}[DATASET]
os.makedirs("/kaggle/working/data", exist_ok=True)
if DATASET == "imagenet_r" and not os.path.exists(DATA):
    !wget -q -O /tmp/imagenet-r.tar https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar && tar -xf /tmp/imagenet-r.tar -C /kaggle/working/data && rm /tmp/imagenet-r.tar
if DATASET == "caltech256" and not os.path.exists(DATA):
    !wget -q -O /tmp/caltech256.tar https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar && tar -xf /tmp/caltech256.tar -C /kaggle/working/data && rm /tmp/caltech256.tar
if DATASET == "cifar100":
    !{PY} -c "from torchvision.datasets import CIFAR100; [CIFAR100('{DATA}', train=t, download=True) for t in (True, False)]"
!ls {DATA} | head -3

In [ ]:
# Tokens come from Kaggle Secrets; they are never written into this notebook.
env = dict(os.environ, WANDB_PROJECT=WANDB_PROJECT, WANDB_DIR="/kaggle/working")
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
env["HF_TOKEN"] = secrets.get_secret("HF_TOKEN").strip()   # required: checkpoints are mirrored to HF_REPO
# Check the token here, before training starts, rather than failing inside train_eval.py.
from huggingface_hub import HfApi
me = HfApi(token=env["HF_TOKEN"]).whoami()   # 401 here = the HF_TOKEN secret is wrong, expired or revoked
spaces = [me["name"]] + [o["name"] for o in me.get("orgs", [])]
assert HF_REPO.split("/")[0] in spaces, f"HF_REPO namespace must be one of {spaces}"
print(f"HF: logged in as {me['name']}, mirroring checkpoints to {HF_REPO}")
try:
    env["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("W&B: online (key from Kaggle Secret)")
except Exception as e:
    env["WANDB_MODE"] = "offline"
    print("W&B: secret WANDB_API_KEY not attached, logging offline:", type(e).__name__)

In [ ]:
import time
LOG = f"/kaggle/working/inf_ssm_{DATASET}_t{TASKS}_seed_{SEED}_{time.strftime('%Y%m%d-%H%M%S')}.log"   # one log per session
cmd = [PY, "-u", "train_eval.py", "-d", DATASET, "-dr", DATA, "-t", str(TASKS),
       "-e", str(EPOCHS), "-b", str(BATCH), "--seed", str(SEED),
       "--cl_method", CL_METHOD, "--finetune", FINETUNE, *(["--full_run"] if FULL_RUN else []),
       "--use_amp", str(USE_AMP), "--data_parallel", str(DATA_PARALLEL), "--grad_ckpt", str(GRAD_CKPT),
       "-jt", str(WORKERS), "--use_wandb", "--wandb_project", WANDB_PROJECT,
       "--progress_interval", str(PROGRESS_INTERVAL),
       "--ckpt_dir", CKPT_DIR, "--hf_repo", HF_REPO,
       "--hf_sync_every", str(HF_SYNC_EVERY), "--hf_squash", str(HF_SQUASH)]
print(" ".join(cmd))
RUN_NAME = None
t0 = time.time(); done = 0
with open(LOG, "w") as f:
    p = subprocess.Popen(cmd, cwd=f"{ROOT}/baselines/inf-ssm", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        print(line, end="", flush=True)   # full log: epochs, val accuracy, per-task results, checkpoints
        if line.startswith(":: run "):
            RUN_NAME = line.split()[2]      # the checkpoint folder / HF subfolder name
        if "Results of task" in line:
            done += 1
            el = time.time() - t0
            print(f"   this session: {done} task(s) in {el/3600:.2f} h, {el/done/3600:.2f} h per task", flush=True)
    p.wait()
print("exit code", p.returncode, f"in {(time.time()-t0)/3600:.2f} h; full log: {LOG}")

In [ ]:
# Keep this session's log next to the checkpoints on HF, then show the results so far.
import json
from huggingface_hub import HfApi
if RUN_NAME:
    try:
        HfApi(token=env["HF_TOKEN"]).upload_file(path_or_fileobj=LOG, path_in_repo=f"{RUN_NAME}/logs/{os.path.basename(LOG)}",
                                                 repo_id=HF_REPO, commit_message="session log")
        print("log uploaded to", f"{HF_REPO}/{RUN_NAME}/logs/")
    except Exception as e:
        print("log upload failed:", e)

res_path = f"{CKPT_DIR}/{RUN_NAME}/results.json"
if RUN_NAME and os.path.exists(res_path):
    r = json.load(open(res_path))
    print(f"tasks done {r['tasks_done']}/{r['num_tasks']} | AA {r['avg_acc']:.2%} | AIA {r['avg_inc_acc']:.2%} | FM {r['forgetting']:.2%} | {r['elapsed_hours']:.2f} h")
    print("AA per task:", " ".join(f"{a:.2%}" for a in r["avg_acc_per_task"]))
    print("paper (CIFAR-100, 10 tasks, seeds 0/10/100): AA 26.53 ± 3.10 | AIA 54.24 ± 1.87 | FM 24.00 ± 1.80")
else:
    print("no results.json yet")

In [ ]:
!tail -n 40 {LOG}